In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
import os

In [ ]:
from matplotlib.patches import Patch, FancyArrow
from matplotlib.legend_handler import HandlerPatch
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter

date_str = "20260803"
output_folder_name = f'png_RS_prelimit_pushing_illustration_{date_str}'
os.makedirs(output_folder_name, exist_ok=True)

plt.rcParams.update({'font.size': 25,
                   'text.usetex': True})

font1 = {'family': 'Arial',
        'weight': 'normal',
        'size': 25}

np.set_printoptions(precision=6)
np.set_printoptions(suppress=True)
np.set_printoptions(threshold=sys.maxsize)


# plot boundary control
plt.figure(figsize = [10.5,6.5])

# for drawing the shading
n = 1
steps = 200
eta = 8.0/9
a = eta / n          # shorthand

# mask the part of the contour outside the state space
t = np.arange(0,7.5*eta/n,1/100/n)
yt0 = np.ones(t.shape)*(-1)
yt1 = np.maximum(0, 2.0*t-10*eta/n)
yt2 = np.minimum(0.5*t+5*eta/n, 7.5*eta/n)
yt3 = np.ones(t.shape)*8.5/n

plt.fill_between(t,yt1,yt2, facecolor=[0.9,0.9,0.9], zorder = 2)

# draw the state space
x = np.linspace(0, 7.5*eta/n, steps, endpoint = True)
plt.plot(x, np.maximum(0,2*x-10*eta/n),'-k', linewidth=3, zorder = 2)
plt.plot(x, np.minimum(7.5*eta/n,0.5*x+5*eta/n), '-k', linewidth=3, zorder = 2)
plt.plot(np.maximum(0,2*x-10*eta/n), x,'-k', linewidth=3, zorder = 2)
plt.plot(np.minimum(7.5*eta/n,0.5*x+5*eta/n), x, '-k', linewidth=3, zorder = 2)

# draw the outer band
epsilon_n = 0.7
x = np.linspace(0, 7.5*eta/n + epsilon_n, steps, endpoint = True)
plt.plot(x, np.maximum(0, 2*(x-epsilon_n*np.sqrt(5)/2)-10*eta/n), color = [0.7, 0.7, 0.7], linewidth=3, zorder = 1)
plt.plot(x, np.minimum(7.5*eta/n + epsilon_n,0.5*x+5*eta/n + epsilon_n*np.sqrt(5)/2), color = [0.7, 0.7, 0.7], linewidth=3, zorder = 1)
plt.plot(np.maximum(0, 2*(x-epsilon_n*np.sqrt(5)/2)-10*eta/n), x, color = [0.7, 0.7, 0.7], linewidth=3, zorder = 1)
plt.plot(np.minimum(7.5*eta/n + epsilon_n,0.5*x+5*eta/n + epsilon_n*np.sqrt(5)/2), x, color = [0.7, 0.7, 0.7], linewidth=3, zorder = 1)

# ===== Classify grid points and color regions A / B =====
gx = np.linspace(0, 10.5*eta/n, 800)
gy = np.linspace(0,  9*eta/n, 800)
GX, GY = np.meshgrid(gx, gy)

# four upper constraints (g <= 0 means satisfied)
g3 = GY - (0.5*GX + 5*a)          # upper left, slope 1/2
g4 = (2*GX - 10*a) - GY           # lower right, slope 2
g5 = GY - 7.5*a                   # top cap
g6 = GX - 7.5*a                   # right cap

# corresponding outer boundaries (pushed out by epsilon_n)
og3 = GY - (0.5*GX + 5*a + epsilon_n*np.sqrt(5)/2)
og4 = (2*(GX - epsilon_n*np.sqrt(5)/2) - 10*a) - GY
og5 = GY - (7.5*a + epsilon_n)
og6 = GX - (7.5*a + epsilon_n)

viol = (g3>0).astype(int)+(g4>0).astype(int)+(g5>0).astype(int)+(g6>0).astype(int)
in_outer = (og3<=0)&(og4<=0)&(og5<=0)&(og6<=0)&(GX>=0)&(GY>=0)

regionA = in_outer & (viol==1)    # violates exactly 1 constraint
regionB = in_outer & (viol>=2)    # violates >= 2 constraints

# colorA = '#bfe3d0'   # light green
colorA = [0.9, 0.95, 1] #'#cfe0ee'  # light blue
colorB = [1, 0.9, 0.9] #'#f6d5d8'   # light pink
if regionA.any():
    plt.contourf(GX, GY, regionA.astype(float), levels=[0.5,1.5],
                 colors=[colorA], zorder=1, antialiased=True)
if regionB.any():
    plt.contourf(GX, GY, regionB.astype(float), levels=[0.5,1.5],
                 colors=[colorB], zorder=1, antialiased=True)

# ===== Dashed extensions of the four upper edges =====
ext = 3 * epsilon_n            # extension length at each end
def draw_ext(P0, P1):
    P0 = np.array(P0, float); P1 = np.array(P1, float)
    u = (P1 - P0); u = u / np.linalg.norm(u)
    A0 = P0 - ext*u; A1 = P1 + ext*u
    plt.plot([A0[0], A1[0]], [A0[1], A1[1]], '--',
             color='0.4', linewidth=3, dashes=(6,4), zorder=3)

draw_ext((5*a, 0),      (7.5*a, 5*a))     # E4 lower right
draw_ext((7.5*a, 5*a),  (7.5*a, 7.5*a))   # E6 right cap
draw_ext((7.5*a, 7.5*a),(5*a, 7.5*a))     # E5 top cap
draw_ext((5*a, 7.5*a),  (0, 5*a))         # E3 upper left
# ==================================================

plt.xlabel(r'$w_1$',font1)
plt.ylabel(r'$w_2$',font1)

labels = [r'$F_1$', r'$F_2$', r'$F_3$', r'$F_4$', r'$F_5$', r'$F_6$']
positions = [
    (0.5 * eta / n, 2.5 * eta / n),  # midpoint of edge 1
    ((2.5+0.4) * eta / n, (6.25-0.4) * eta / n),  # midpoint of edge 2
    (6 * eta / n, (7.5-0.5) * eta / n),  # midpoint of edge 3
    ((7.5-0.5) * eta / n, 6 * eta / n),  # midpoint of edge 4
    ((6.25-0.4) * eta / n, (2.5+0.4) * eta / n),  # midpoint of edge 5
    (2.5 * eta / n, 0.5 * eta / n)  # point 6 (adjusted)
]
for label, (x, y) in zip(labels, positions):
    plt.text(x, y, label, fontsize = 24, ha='center', va='center')

# ===== Four blue bands E_1..E_4 (clockwise: upper left -> top -> right -> lower right) =====
E_labels = [r'$E_1$', r'$E_2$', r'$E_3$', r'$E_4$']
E_positions = [
    (2.5 * a - 0.25/np.sqrt(5), 6.25 * a + 0.5/np.sqrt(5) + 0.1),   # E_1: F2 blue band (upper left, slope 1/2)
    (6.25 * a + 0.4 + 0.05,                  7.5 * a + 0.25 + 0.1),              # E_2: F3 blue band (top cap)
    (7.5 * a + 0.25 + 0.1,            6.25 * a + 0.4 + 0.05),                    # E_3: F4 blue band (right cap)
    (6.25 * a + 0.5/np.sqrt(5) + 0.1, 2.5 * a - 0.25/np.sqrt(5)),   # E_4: F5 blue band (lower right, slope 2)
]
for label, (x, y) in zip(E_labels, E_positions):
    plt.text(x, y, label, color = [0.3, 0.5, 1], fontsize = 24, ha='center', va='center')

# ===== Three pink corner regions C_1..C_3 (clockwise: upper-left -> upper-right -> right corner) =====
C_labels = [r'$C_1$', r'$C_2$', r'$C_3$']
C_positions = [
    (5 * a - 0.08,    7.5 * a + 0.35),    # C_1: F2∩F3 corner (upper-left)
    (7.5 * a + 0.35, 7.5 * a + 0.35),   # C_2: F3∩F4 corner (upper-right)
    (7.5 * a + 0.35,  5 * a - 0.08),      # C_3: F4∩F5 corner (right)
]
for label, (x, y) in zip(C_labels, C_positions):
    plt.text(x, y, label, color = [1, 0.4, 0.4], fontsize = 24, ha='center', va='center')
# ==================================================

# Custom legend
custom_legend = [
    Patch(facecolor=[0.9,0.9,0.9], edgecolor='black', label=r'State space'),
    Patch(facecolor='none', edgecolor=[0.7, 0.7, 0.7], label=r'Enlarged state space'),
    Patch(facecolor=colorA, edgecolor='none', label=r'Face-interior band subsets'),
    Patch(facecolor=colorB, edgecolor='none', label=r'Corner band subsets'),
]

plt.legend(handles=custom_legend, loc = 'center left', bbox_to_anchor=(0.95, 0.5), fontsize = 23)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().xaxis.set_ticks_position('bottom')
plt.gca().yaxis.set_ticks_position('left')

plt.xlim([0, 9*eta/n])
plt.ylim([0, 9*eta/n])
plt.xticks([])
plt.yticks([])

plt.tight_layout()
plt.grid(False)

# Save the figure as a PDF file
plt.savefig(f'{output_folder_name}/RS_prelimit_state_space_{date_str}.pdf', format='pdf', bbox_inches='tight', pad_inches=0.05)